# 🧬 GroupDNA — WhatsApp Group Analytics
---
**Project**: GroupDNA — Your WhatsApp Group, Decoded  
**Built with**: Python Fundamentals + NumPy only (no pandas, no matplotlib, no regex)  
**Dataset**: hostel_bois.txt  
**Platform**: Google Colab  

> *"Spotify Wrapped, but for your friend group."*

---
**Name** - Alzafar Khan  
**Batch** - june 1  
**Submission Date** - 28-06-2026



## 📦 Cell 1 — Imports & Configuration

In [10]:
import numpy as np
from datetime import datetime

# ── FILE PATH
FILE_PATH = "/content/hostel_bois.txt"

# ── STOP WORDS
# Common words excluded from word-frequency analysis
STOP_WORDS = {
    "i", "is", "the", "a", "and", "or", "to", "of", "in", "on", "for",
    "it", "this", "that", "was", "are", "be", "with", "as", "at", "by",
    "an", "we", "so", "if", "do", "not", "me", "my", "he", "she", "you",
    "they", "we", "but", "its", "our", "up", "out", "no", "can", "just",
    "have", "all", "will", "from", "your", "had", "has", "what", "which",
    "there", "been", "were", "their", "more", "also", "any", "would",
    "who", "him", "her", "his", "them", "then", "than", "too", "very",
    "when", "how", "about", "one", "get", "like", "now", "did", "go",
    "know", "got", "oh", "ok", "okay", "yeah", "yes", "haha", "lol",
    "u", "r", "n", "hi", "hey", "na", "nahi", "aur", "main", "mein",
    "hai", "ho", "ka", "ki", "ke", "se", "ko", "ne", "toh", "ab",
    "bhi", "kuch", "tum", "aap", "woh", "ye", "vo", "ek", "hain",
    "nhi", "nhi", "koi", "kar", "tha", "the", "raha", "rahi", "gaya",
    "lo", "le", "de", "aa", "ja", "kal", "agar", "phir", "fir", "wala",
    "pe", "par", "mat", "kab", "kaise", "kyun", "mujhe", "tumhe",
    "unhe", "unka", "inke", "hoga", "hogi", "hua", "hui", "use",
    "message", "deleted", "media", "omitted", "edited"
}

# ── CARING KEYWORDS
# Used for "Group Mom" archetype detection
CARING_KEYWORDS = [
    "okay", "safe", "eat", "sleep", "take care", "are you",
    "please", "reminder", "drink water", "don't forget",
    "doing okay", "feeling", "better", "rest", "careful",
    "anyone needs", "help", "everyone", "study", "notes",
    "call home", "doing well", "health", "calm", "don't worry"
]

# ── COMEDY KEYWORDS
COMEDY_KEYWORDS = ["lol", "lmao", "haha", "rofl", "lmfao", "hahaha", "hehe", "xd"]

# ── PUNCTUATION STRIP SET
PUNCTUATION = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~\u0964')

print("✅  Configuration loaded. Ready to parse.")

✅  Configuration loaded. Ready to parse.


## 🔍 Feature 1 — The Chat Parser

In [23]:

#  FEATURE 1 — CHAT PARSER
#  Reads hostel_bois.txt, extracts timestamp / sender / text
#  Handles: system msgs, media omitted, deleted msgs, multi-line


def is_date_line(line):
    """
    Returns True if the line starts with a WhatsApp timestamp pattern.
    Pattern: DD/MM/YY, HH:MM
    We check the first 8 characters manually (no regex allowed).
    """
    if len(line) < 8:
        return False
    # Chars 0-1 = day digits, 2 = '/', 3-4 = month, 5 = '/', 6-7 = year
    return (line[2] == '/' and line[5] == '/' and
            line[0].isdigit() and line[1].isdigit() and
            line[3].isdigit() and line[4].isdigit() and
            line[6].isdigit() and line[7].isdigit())


def parse_line(line):
    """
    Parses a single WhatsApp message line.
    Returns a dict with keys: timestamp_str, sender, text
    Returns None for system messages (no sender colon found).
    """
    # Split at ' - ' once to separate timestamp from the rest
    parts = line.split(' - ', 1)
    if len(parts) < 2:
        return None

    timestamp_str = parts[0].strip()   # e.g. "12/04/24, 23:14"
    rest = parts[1].strip()            # e.g. "Rahul: scene fix"

    # System messages have no ': ' after sender
    if ': ' not in rest:
        return None  # system message

    # Split sender from message
    sender, text = rest.split(': ', 1)
    sender = sender.strip()
    text = text.strip()

    return {
        "timestamp_str": timestamp_str,
        "sender": sender,
        "text": text
    }


def parse_chat(filepath):
    """
    Main parser. Reads the file and builds a list of message dicts.

    Returns:
        messages      – list of dicts {timestamp_str, sender, text,
                                        is_media, is_deleted, datetime_obj}
        system_count  – count of system messages skipped
        media_count   – count of <Media omitted> entries
        deleted_count – count of deleted messages
        group_name    – extracted from system messages if present
    """
    messages      = []
    system_count  = 0
    media_count   = 0
    deleted_count = 0
    group_name    = "WhatsApp Group"
    current_message = None   # Holds the message currently being built (potentially multi-line)

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            raw_lines = f.read().split('\n')
    except FileNotFoundError:
        print(f"❌  File not found: {filepath}")
        print("    Upload hostel_bois.txt to /content/ in Colab.")
        return [], 0, 0, 0, "Unknown"

    for line in raw_lines:
        line = line.strip()

        # Skip blank lines
        if not line:
            continue

        if is_date_line(line):
            # A new message starts. Finalize the previous message if it exists.
            if current_message is not None:
                # Classify and process the completed message
                text = current_message["text"]
                is_media   = text == "<Media omitted>"
                is_deleted = text in ("This message was deleted", "You deleted this message")

                current_message["is_media"]   = is_media
                current_message["is_deleted"] = is_deleted

                # Parse datetime object
                try:
                    current_message["datetime_obj"] = datetime.strptime(current_message["timestamp_str"], '%d/%m/%y, %H:%M')
                except ValueError:
                    try:
                        current_message["datetime_obj"] = datetime.strptime(current_message["timestamp_str"], '%d/%m/%Y, %H:%M')
                    except ValueError:
                        current_message["datetime_obj"] = None

                if is_media:
                    media_count += 1
                if is_deleted:
                    deleted_count += 1
                messages.append(current_message)

            # Now, parse the current line as the start of a new message.
            parsed_new_message = parse_line(line)

            if parsed_new_message is None:
                # System message
                system_count += 1
                # Try to extract group name
                if 'created group' in line and '"' in line:
                    start = line.index('"') + 1
                    end   = line.rindex('"')
                    group_name = line[start:end]
                current_message = None  # System messages don't have continuations
            else:
                current_message = parsed_new_message  # This message might be multi-line
        else:
            # This is a continuation line for the current message
            if current_message is not None:
                current_message["text"] += " " + line
            # else: this line is junk or an unhandled multi-line without a preceding message, ignore it.

    # After the loop, add the last current_message if it exists
    if current_message is not None:
        # Classify and process the final message
        text = current_message["text"]
        is_media   = text == "<Media omitted>"
        is_deleted = text in ("This message was deleted", "You deleted this message")

        current_message["is_media"]   = is_media
        current_message["is_deleted"] = is_deleted

        try:
            current_message["datetime_obj"] = datetime.strptime(current_message["timestamp_str"], '%d/%m/%y, %H:%M')
        except ValueError:
            try:
                current_message["datetime_obj"] = datetime.strptime(current_message["timestamp_str"], '%d/%m/%Y, %H:%M')
            except ValueError:
                current_message["datetime_obj"] = None

        if is_media:
            media_count += 1
        if is_deleted:
            deleted_count += 1
        messages.append(current_message)

    return messages, system_count, media_count, deleted_count, group_name


# ── RUN PARSER
messages, sys_count, med_count, del_count, GROUP_NAME = parse_chat(FILE_PATH)

# Quick counts
total_parsed = len(messages)
senders      = list({m["sender"] for m in messages})
real_messages = [m for m in messages if not m["is_media"] and not m["is_deleted"]]

# Date range
dates_with_dt = [m["datetime_obj"] for m in messages if m["datetime_obj"]]
first_dt = min(dates_with_dt) if dates_with_dt else None
last_dt  = max(dates_with_dt) if dates_with_dt else None
total_days = (last_dt.date() - first_dt.date()).days + 1 if first_dt and last_dt else 0

print(f"✅  Successfully parsed {total_parsed:,} messages")
print(f"    Participants   : {len(senders)}")
print(f"    Date range     : {first_dt.strftime('%d %B %Y')} → {last_dt.strftime('%d %B %Y')}")
print(f"    Total days     : {total_days}")
print(f"    System msgs    : {sys_count}")
print(f"    Media omitted  : {med_count}")
print(f"    Deleted msgs   : {del_count}")
print()
print("  First 3 messages:")
for m in messages[:3]:
    print(f"    [{m['timestamp_str']}] {m['sender']}: {m['text'][:60]}")
print("  Last 3 messages:")
for m in messages[-3:]:
    print(f"    [{m['timestamp_str']}] {m['sender']}: {m['text'][:60]}")

✅  Successfully parsed 3,174 messages
    Participants   : 6
    Date range     : 01 April 2024 → 30 May 2024
    Total days     : 60
    System msgs    : 4
    Media omitted  : 32
    Deleted msgs   : 15

  First 3 messages:
    [01/04/24, 01:17] Rahul: scene fix
    [01/04/24, 01:17] Rahul: haan
    [01/04/24, 01:18] Rahul: kya scene
  Last 3 messages:
    [30/05/24, 21:17] Aman: the existential dread is back
    [30/05/24, 21:30] Karan: Long day guys, woke up at six for that placement workshop wh
    [30/05/24, 23:31] Aman: anyone awake?


## 📊 Feature 2 — Group Overview & Participant Statistics

In [12]:

#  FEATURE 2 — GROUP OVERVIEW & PARTICIPANT STATISTICS
#  Message counts, percentages, per-person word counts,
#  avg message length — all using pure Python dicts


def build_participant_stats(messages):
    """
    Builds per-participant statistics dict.
    Returns a dict keyed by sender name with:
        msg_count, word_count, char_count, media_count, deleted_count
    """
    stats = {}

    for m in messages:
        sender = m["sender"]
        if sender not in stats:
            stats[sender] = {
                "msg_count"    : 0,
                "word_count"   : 0,
                "char_count"   : 0,
                "media_count"  : 0,
                "deleted_count": 0,
            }

        stats[sender]["msg_count"] += 1

        if m["is_media"]:
            stats[sender]["media_count"] += 1
        elif m["is_deleted"]:
            stats[sender]["deleted_count"] += 1
        else:
            words = m["text"].split()
            stats[sender]["word_count"] += len(words)
            stats[sender]["char_count"] += len(m["text"])

    return stats


def make_bar(value, max_value, width=20):
    """Creates a filled Unicode block bar proportional to value/max_value."""
    if max_value == 0:
        return "." * width
    filled = int((value / max_value) * width)
    return "█" * filled + "░" * (width - filled) if filled > 0 else "." + "░" * (width - 1)


def print_group_overview(messages, stats, group_name, first_dt, last_dt, total_days,
                          sys_count, med_count, del_count):
    """Prints the Group Overview section of the final report."""
    total_msgs = len(messages)
    max_msgs   = max(s["msg_count"] for s in stats.values())

    # Sort participants by message count descending
    sorted_people = sorted(stats.items(), key=lambda x: x[1]["msg_count"], reverse=True)

    W = 66
    print("╔" + "═" * W + "╗")
    print("║" + "  🧬  GROUPDNA — WhatsApp Group Analytics Report  🧬".center(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  {'Group Name':<18}: {group_name:<44}║")
    print(f"║  {'Period':<18}: {first_dt.strftime('%d %B %Y')} → {last_dt.strftime('%d %B %Y')} ({total_days} days){'':>6}║")
    print(f"║  {'Total Messages':<18}: {total_msgs:,}{'':<41}║")
    print(f"║  {'Participants':<18}: {len(stats)}{'':<45}║")
    print(f"║  {'System Messages':<18}: {sys_count}{'':<45}║")
    print(f"║  {'Media Shared':<18}: {med_count}{'':<45}║")
    print(f"║  {'Deleted Messages':<18}: {del_count}{'':<45}║")
    print("╠" + "═" * W + "╣")
    print("║" + "  📩  MESSAGES PER PERSON".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    rank_icons = ["🥇", "🥈", "🥉", "4️⃣ ", "5️⃣ ", "6️⃣ ", "7️⃣ ", "8️⃣ "]
    for i, (name, s) in enumerate(sorted_people):
        count = s["msg_count"]
        pct   = count / total_msgs * 100
        bar   = make_bar(count, max_msgs, 18)
        icon  = rank_icons[i] if i < len(rank_icons) else f"{i+1}. "
        real_msg_count = count - s["media_count"] - s["deleted_count"]
        avg_words = (s["word_count"] / real_msg_count) if real_msg_count > 0 else 0
        print(f"║  {icon} {name:<10} {bar}  {count:>4} ({pct:>4.1f}%)  avg {avg_words:.1f} wds/msg  ║")

    print("╚" + "═" * W + "╝")


# ── BUILD AND DISPLAY
participant_stats = build_participant_stats(messages)
print_group_overview(messages, participant_stats, GROUP_NAME,
                     first_dt, last_dt, total_days,
                     sys_count, med_count, del_count)


╔══════════════════════════════════════════════════════════════════╗
║          🧬  GROUPDNA — WhatsApp Group Analytics Report  🧬        ║
╠══════════════════════════════════════════════════════════════════╣
║  Group Name        : Hostel Bois 4ever                           ║
║  Period            : 01 April 2024 → 30 May 2024 (60 days)      ║
║  Total Messages    : 3,174                                         ║
║  Participants      : 6                                             ║
║  System Messages   : 4                                             ║
║  Media Shared      : 32                                             ║
║  Deleted Messages  : 15                                             ║
╠══════════════════════════════════════════════════════════════════╣
║  📩  MESSAGES PER PERSON                                          ║
╠══════════════════════════════════════════════════════════════════╣
║  🥇 Rahul      ██████████████████   953 (30.0%)  avg 2.6 wds/msg  ║
║  🥈 Priya      ███████

## 📅 Feature 3 — Most Active Day & Hour Analysis

In [13]:

#  FEATURE 3 — DATE & HOUR ANALYSIS
#  Busiest single day, busiest hour, daily + weekly breakdown


def build_date_hour_counts(messages):
    """
    Returns:
        date_counts – dict {date_str: count}
        hour_counts – dict {hour_int: count}
        weekday_counts – dict {weekday_name: count}
    """
    date_counts    = {}
    hour_counts    = {h: 0 for h in range(24)}
    weekday_counts = {}
    weekday_names  = ["Monday", "Tuesday", "Wednesday", "Thursday",
                       "Friday", "Saturday", "Sunday"]

    for m in messages:
        dt = m["datetime_obj"]
        if dt is None:
            continue
        date_key    = dt.strftime("%d/%m/%Y")
        weekday_key = weekday_names[dt.weekday()]
        hour_key    = dt.hour

        date_counts[date_key]       = date_counts.get(date_key, 0) + 1
        hour_counts[hour_key]      += 1
        weekday_counts[weekday_key] = weekday_counts.get(weekday_key, 0) + 1

    return date_counts, hour_counts, weekday_counts


def format_date_readable(date_str):
    """Converts DD/MM/YYYY to '14 April 2024'."""
    try:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return dt.strftime("%d %B %Y")
    except ValueError:
        return date_str


def print_date_hour_analysis(date_counts, hour_counts, weekday_counts, total_days):
    """Prints busiest day, busiest hour, weekday breakdown."""
    busiest_day   = max(date_counts, key=date_counts.get)
    busiest_hour  = max(hour_counts, key=hour_counts.get)
    avg_per_hour  = {h: hour_counts[h] / total_days for h in range(24)}
    max_hour_avg  = max(avg_per_hour[h] for h in range(24))

    # Weekday sort: Mon–Sun
    weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    max_wd_count  = max(weekday_counts.values()) if weekday_counts else 1

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  📅  DATE & ACTIVITY ANALYSIS".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  🔥 Busiest Day  : {format_date_readable(busiest_day):<20} ({date_counts[busiest_day]} messages){'':>6}║")
    print(f"║  ⏰ Busiest Hour : {busiest_hour:02d}:00 – {busiest_hour+1:02d}:00{'':>34}║")
    print(f"║     (avg {avg_per_hour[busiest_hour]:.1f} messages/day during that hour){'':>27}║")
    print("╠" + "═" * W + "╣")
    print("║  📊  HOURLY DISTRIBUTION (all messages, 24-hour)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    max_h = max(hour_counts.values()) or 1
    for h in range(0, 24, 3):
        row = f"  {h:02d}h: "
        for hh in range(h, min(h+3, 24)):
            bar_len = int(hour_counts[hh] / max_h * 12)
            bar = "█" * bar_len + "░" * (12 - bar_len)
            row += f" {hh:02d}▕{bar}▏{hour_counts[hh]:>4}  "
        print(f"║{row.ljust(W)}║")

    print("╠" + "═" * W + "╣")
    print("║  📆  WEEKDAY BREAKDOWN".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    for wd in weekday_order:
        count   = weekday_counts.get(wd, 0)
        bar_len = int(count / max_wd_count * 20)
        bar     = "█" * bar_len + "░" * (20 - bar_len)
        print(f"║  {wd:<10}  {bar}  {count:>5} msgs{'':>9}║")
    print("╚" + "═" * W + "╝")


# ── RUN
date_counts, hour_counts, weekday_counts = build_date_hour_counts(messages)
print_date_hour_analysis(date_counts, hour_counts, weekday_counts, total_days)

# Store busiest_hour globally for archetype detection later
BUSIEST_HOUR = max(hour_counts, key=hour_counts.get)
print(f"\n✅  Date analysis complete. Busiest hour: {BUSIEST_HOUR:02d}:00")



╔══════════════════════════════════════════════════════════════════╗
║  📅  DATE & ACTIVITY ANALYSIS                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  🔥 Busiest Day  : 04 May 2024          (76 messages)      ║
║  ⏰ Busiest Hour : 18:00 – 19:00                                  ║
║     (avg 4.1 messages/day during that hour)                           ║
╠══════════════════════════════════════════════════════════════════╣
║  📊  HOURLY DISTRIBUTION (all messages, 24-hour)                 ║
╠══════════════════════════════════════════════════════════════════╣
║  00h:  00▕██░░░░░░░░░░▏  57   01▕███░░░░░░░░░▏  82   02▕████░░░░░░░░▏  83  ║
║  03h:  03▕███░░░░░░░░░▏  77   04▕█████░░░░░░░▏ 110   05▕█░░░░░░░░░░░▏  29  ║
║  06h:  06▕█░░░░░░░░░░░▏  33   07▕██░░░░░░░░░░▏  55   08▕█████░░░░░░░▏ 122  ║
║  09h:  09▕███████░░░░░▏ 151   10▕███████░░░░░▏ 160   11▕█████░░░░░░░▏ 114  ║
║  12h:  12▕█████████░░░▏ 193   13▕███████░░░░░▏ 159   14▕███████

## 🔥 Feature 4 — NumPy Activity Heatmap

In [14]:
#  FEATURE 4 — NUMPY ACTIVITY HEATMAP
#  Builds a (participants × 24 hours) matrix using np.zeros
#  Renders with Unicode block characters at 4 shade levels

def build_activity_matrix(messages, participant_list):
    """
    Creates a NumPy matrix: rows = participants, cols = hours (0-23).
    Each cell = total messages sent by that person at that hour.
    Also builds:
        daily_matrix   – (participants × total_days) for per-day analysis
        hourly_total   – 1D array, total msgs per hour across all people
    """
    n_people = len(participant_list)
    person_idx = {name: i for i, name in enumerate(participant_list)}

    # Primary heatmap: person × hour
    heatmap_matrix = np.zeros((n_people, 24), dtype=int)

    # Daily activity matrix
    all_dates = sorted({m["datetime_obj"].date() for m in messages
                        if m["datetime_obj"]})
    date_idx  = {d: i for i, d in enumerate(all_dates)}
    daily_matrix = np.zeros((n_people, len(all_dates)), dtype=int)

    for m in messages:
        dt = m["datetime_obj"]
        if dt is None:
            continue
        pidx = person_idx.get(m["sender"])
        if pidx is None:
            continue
        heatmap_matrix[pidx, dt.hour] += 1
        didx = date_idx.get(dt.date())
        if didx is not None:
            daily_matrix[pidx, didx] += 1

    hourly_total = np.sum(heatmap_matrix, axis=0)

    return heatmap_matrix, daily_matrix, all_dates, hourly_total


def render_heatmap(heatmap_matrix, participant_list):
    """
    Renders the (n × 24) heatmap with 4 Unicode shading levels:
      0 %     → '.'
      1–25%   → '░'
      25–50%  → '▒'
      50–75%  → '▓'
      75–100% → '█'
    Each person's row is normalised by their own max, so even low-volume
    members show up meaningfully.
    """
    SHADES = ['.', '░', '▒', '▓', '█']

    def value_to_shade(val, row_max):
        if row_max == 0 or val == 0:
            return '.'
        ratio = val / row_max
        if ratio <= 0.25:
            return '░'
        elif ratio <= 0.50:
            return '▒'
        elif ratio <= 0.75:
            return '▓'
        else:
            return '█'

    W      = 74
    header = "  00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23"
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  🌡️   ACTIVITY HEATMAP  (messages per hour of day)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    print("║" + f"  {'NAME':<12}{header}".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    for i, name in enumerate(participant_list):
        row     = heatmap_matrix[i]
        row_max = int(np.max(row))
        shade_row = "  ".join(value_to_shade(row[h], row_max) for h in range(24))
        line    = f"  {name:<12}  {shade_row}"
        print(f"║{line.ljust(W)}║")

    print("╠" + "═" * W + "╣")
    print("║  LEGEND:  . = none  ░ = low  ▒ = medium  ▓ = high  █ = peak".ljust(W) + "║")
    print("╚" + "═" * W + "╝")


def print_numpy_stats(heatmap_matrix, participant_list):
    """Prints interesting aggregate NumPy stats from the matrix."""
    print()
    print("📐  NumPy Matrix Stats:")
    print(f"    Shape            : {heatmap_matrix.shape}  (participants × hours)")
    print(f"    Total msgs (sum) : {int(np.sum(heatmap_matrix)):,}")
    print(f"    Peak single cell : {int(np.max(heatmap_matrix))} msgs")
    peak_idx = np.unravel_index(np.argmax(heatmap_matrix), heatmap_matrix.shape)
    print(f"    Peak belongs to  : {participant_list[peak_idx[0]]} at {peak_idx[1]:02d}:00h")

    print()
    print("    Row sums (total msgs per person via NumPy):")
    row_sums = np.sum(heatmap_matrix, axis=0)   # col sums
    person_sums = np.sum(heatmap_matrix, axis=1) # row sums
    for i, name in enumerate(participant_list):
        print(f"      {name:<10}: {int(person_sums[i]):>5} msgs")

    print()
    night_hours  = list(range(23, 24)) + list(range(0, 5))
    night_matrix = heatmap_matrix[:, night_hours]
    night_pct    = np.sum(night_matrix, axis=1) / (np.sum(heatmap_matrix, axis=1) + 1e-9) * 100

    print("    Night activity % (23:00 – 04:59):")
    for i, name in enumerate(participant_list):
        bar = "█" * int(night_pct[i] / 5) + "░" * (20 - int(night_pct[i] / 5))
        print(f"      {name:<10}: {bar} {night_pct[i]:.1f}%")

    return night_pct


# ── RUN
PARTICIPANT_LIST = sorted(participant_stats.keys(),
                          key=lambda n: participant_stats[n]["msg_count"],
                          reverse=True)

HEATMAP_MATRIX, DAILY_MATRIX, ALL_DATES, HOURLY_TOTAL = build_activity_matrix(
    messages, PARTICIPANT_LIST
)

render_heatmap(HEATMAP_MATRIX, PARTICIPANT_LIST)
NIGHT_PCT = print_numpy_stats(HEATMAP_MATRIX, PARTICIPANT_LIST)
print("\n✅  NumPy heatmap complete.")



╔══════════════════════════════════════════════════════════════════════════╗
║  🌡️   ACTIVITY HEATMAP  (messages per hour of day)                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║  NAME          00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23║
╠══════════════════════════════════════════════════════════════════════════╣
║  Rahul         ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓║
║  Priya         .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░║
║  Neha          .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒║
║  Aman          ▓  █  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓║
║  Karan         .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░║
║  Vikas         .  .  .  .  .  .  .  ▒  █  ▒  ▒  .  ▓  ▓  .  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓║
╠══════════════════════════════════════════════════════════════════

## 💬 Feature 5 — Word Frequency Analysis

In [15]:
#  FEATURE 5 — WORD FREQUENCY ANALYSIS
#  Tokenises all real messages, strips stopwords & punctuation,
#  shows top words globally and per person with visual bars.
#  Uses plain dict (no collections.Counter allowed).

def strip_punctuation(word):
    """Removes leading/trailing punctuation from a word."""
    while word and word[0] in PUNCTUATION:
        word = word[1:]
    while word and word[-1] in PUNCTUATION:
        word = word[:-1]
    return word


def tokenize(text):
    """
    Lowercases the text, splits into tokens, strips punctuation,
    removes stop words and very short tokens.
    Returns a list of clean word strings.
    """
    words = []
    for raw in text.lower().split():
        word = strip_punctuation(raw)
        if len(word) < 2:
            continue
        if word in STOP_WORDS:
            continue
        if word.isdigit():
            continue
        words.append(word)
    return words


def build_word_freq(messages):
    """
    Builds two dicts:
        global_freq – {word: count} across all messages
        person_freq – {sender: {word: count}}
    Excludes media-omitted and deleted messages.
    """
    global_freq = {}
    person_freq = {}

    for m in messages:
        if m["is_media"] or m["is_deleted"]:
            continue
        sender = m["sender"]
        if sender not in person_freq:
            person_freq[sender] = {}

        for word in tokenize(m["text"]):
            global_freq[word] = global_freq.get(word, 0) + 1
            person_freq[sender][word] = person_freq[sender].get(word, 0) + 1

    return global_freq, person_freq


def top_n(freq_dict, n=10):
    """Returns top-n (word, count) pairs sorted by count desc."""
    sorted_items = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)
    return sorted_items[:n]


def print_word_analysis(global_freq, person_freq, participant_list):
    """Prints global top words and per-person top-5 words with bars."""
    top_global  = top_n(global_freq, 15)
    max_count   = top_global[0][1] if top_global else 1
    total_words = sum(global_freq.values())

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  💬  THIS GROUP'S FAVOURITE WORDS  🗣️".center(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  {'Total unique words':<25}: {len(global_freq):>6}{'':>27}║")
    print(f"║  {'Total word occurrences':<25}: {total_words:>6}{'':>27}║")
    print("╠" + "═" * W + "╣")
    print(f"║  {'RANK':<5} {'WORD':<15} {'BAR':<20} {'COUNT':>6} {'%':>6}  ║")
    print("╠" + "═" * W + "╣")

    for rank, (word, count) in enumerate(top_global, 1):
        bar_len  = int(count / max_count * 18)
        bar      = "█" * bar_len + "░" * (18 - bar_len)
        pct      = count / total_words * 100
        emoji    = "🔥" if rank == 1 else "  "
        print(f"║  {emoji}{rank:<3} {word:<15} {bar}  {count:>5}  {pct:>5.2f}%  ║")

    print("╠" + "═" * W + "╣")
    print("║  📌  TOP 5 WORDS PER PERSON".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    for name in participant_list:
        pfreq   = person_freq.get(name, {})
        top5    = top_n(pfreq, 5)
        words_str = "  ".join(f"{w}({c})" for w, c in top5)
        print(f"║  👤 {name:<10}  {words_str[:54].ljust(54)}║")

    print("╚" + "═" * W + "╝")


# ── RUN
GLOBAL_FREQ, PERSON_FREQ = build_word_freq(messages)
print_word_analysis(GLOBAL_FREQ, PERSON_FREQ, PARTICIPANT_LIST)
print("\n✅  Word frequency analysis complete.")



╔══════════════════════════════════════════════════════════════════╗
║                💬  THIS GROUP'S FAVOURITE WORDS  🗣️               ║
╠══════════════════════════════════════════════════════════════════╣
║  Total unique words       :    501                           ║
║  Total word occurrences   :  17325                           ║
╠══════════════════════════════════════════════════════════════════╣
║  RANK  WORD            BAR                   COUNT      %  ║
╠══════════════════════════════════════════════════════════════════╣
║  🔥1   guys            ██████████████████    318   1.84%  ║
║    2   am              ██████████████░░░░    260   1.50%  ║
║    3   today           ██████████████░░░░    257   1.48%  ║
║    4   everyone        ██████████░░░░░░░░    187   1.08%  ║
║    5   telling         ██████████░░░░░░░░    179   1.03%  ║
║    6   bhai            █████████░░░░░░░░░    160   0.92%  ║
║    7   started         ████████░░░░░░░░░░    150   0.87%  ║
║    8   scene           ███

## ⏱️ Feature 6 — Response Speed & Silent Streaks

In [16]:

#  FEATURE 6 — RESPONSE TIMES & SILENT STREAKS
#  Uses datetime arithmetic to find avg reply speed per person
#  and the longest continuous silence (days with 0 messages).

def compute_response_times(messages):
    """
    For each message, if the sender is different from the previous message's
    sender, records the gap as a potential response time for the new sender.
    Returns: {sender: [gap_in_seconds, ...]}
    """
    reply_times = {name: [] for name in PARTICIPANT_LIST}

    for i in range(1, len(messages)):
        curr = messages[i]
        prev = messages[i - 1]

        # Only count if different sender and both have valid datetimes
        if (curr["sender"] != prev["sender"] and
                curr["datetime_obj"] and prev["datetime_obj"]):
            gap = (curr["datetime_obj"] - prev["datetime_obj"]).total_seconds()
            # Ignore gaps longer than 12 hours (new conversation, not a reply)
            if 0 < gap < 43200:
                if curr["sender"] in reply_times:
                    reply_times[curr["sender"]].append(gap)

    return reply_times


def compute_silent_streaks(messages, all_dates, participant_list):
    """
    For each participant, finds:
        longest_streak    – int, max consecutive days with 0 messages
        streak_start/end  – the date range of the longest streak
        total_silent_days – total days with 0 messages
    """
    # Build set of active dates per person
    active_dates = {name: set() for name in participant_list}
    for m in messages:
        if m["datetime_obj"]:
            active_dates[m["sender"]].add(m["datetime_obj"].date())

    streaks = {}
    for name in participant_list:
        active = active_dates[name]
        max_streak   = 0
        streak_start = None
        streak_end   = None
        current      = 0
        curr_start   = None
        silent_total = 0

        for d in all_dates:
            if d not in active:
                silent_total += 1
                if current == 0:
                    curr_start = d
                current += 1
                if current > max_streak:
                    max_streak   = current
                    streak_start = curr_start
                    streak_end   = d
            else:
                current = 0

        streaks[name] = {
            "max_streak"  : max_streak,
            "streak_start": streak_start,
            "streak_end"  : streak_end,
            "silent_total": silent_total,
            "silent_pct"  : silent_total / len(all_dates) * 100,
        }

    return streaks


def print_response_analysis(reply_times, streaks, participant_list):
    """Prints response time and silent streak sections."""

    # Compute averages
    avg_times = {}
    for name in participant_list:
        times = reply_times[name]
        avg_times[name] = sum(times) / len(times) if times else float('inf')

    valid = {n: t for n, t in avg_times.items() if t != float('inf')}
    if valid:
        fastest = min(valid, key=valid.get)
        slowest = max(valid, key=valid.get)
    else:
        fastest = slowest = participant_list[0]

    def fmt_time(secs):
        if secs == float('inf'):
            return "N/A"
        if secs < 60:
            return f"{secs:.0f} sec"
        elif secs < 3600:
            return f"{secs/60:.1f} min"
        else:
            return f"{secs/3600:.1f} hrs"

    sorted_by_streak = sorted(participant_list,
                               key=lambda n: streaks[n]["max_streak"],
                               reverse=True)

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  ⏱️   RESPONSE PATTERNS & SILENT STREAKS".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  ⚡ Fastest Replier : {fastest:<10} (avg {fmt_time(avg_times.get(fastest,0))}){'':>19}║")
    print(f"║  🐢 Slowest Replier : {slowest:<10} (avg {fmt_time(avg_times.get(slowest,0))}){'':>19}║")
    print("╠" + "═" * W + "╣")
    print("║  📊  AVG REPLY TIMES PER PERSON".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    max_valid = max((t for t in avg_times.values() if t != float('inf')), default=1)
    for name in participant_list:
        t   = avg_times.get(name, float('inf'))
        bar_len = int(min(t, max_valid) / max_valid * 18) if t != float('inf') else 18
        bar = "█" * bar_len + "░" * (18 - bar_len)
        print(f"║  {name:<10}  {bar}  {fmt_time(t):<12}{'':>8}║")

    print("╠" + "═" * W + "╣")
    print("║  🏜️   LONGEST SILENT STREAKS (consecutive days, 0 messages)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    for name in sorted_by_streak:
        s    = streaks[name]
        ms   = s["max_streak"]
        sp   = s["silent_pct"]
        rng  = ""
        if s["streak_start"] and s["streak_end"] and ms > 0:
            rng = f"({s['streak_start'].strftime('%d %b')} – {s['streak_end'].strftime('%d %b')})"
        bar_len = int(ms / max(st["max_streak"] for st in streaks.values()) * 18) if                   max(st["max_streak"] for st in streaks.values()) > 0 else 0
        bar = "█" * bar_len + "░" * (18 - bar_len)
        print(f"║  {name:<10}  {bar}  {ms:>2} days  {rng:<18}({sp:.0f}% silent)║")

    print("╚" + "═" * W + "╝")

    return avg_times, streaks


# ── RUN
REPLY_TIMES = compute_response_times(messages)
STREAKS     = compute_silent_streaks(messages, ALL_DATES, PARTICIPANT_LIST)
AVG_TIMES, _ = print_response_analysis(REPLY_TIMES, STREAKS, PARTICIPANT_LIST)
print("\n✅  Response analysis complete.")



╔══════════════════════════════════════════════════════════════════╗
║  ⏱️   RESPONSE PATTERNS & SILENT STREAKS                         ║
╠══════════════════════════════════════════════════════════════════╣
║  ⚡ Fastest Replier : Rahul      (avg 36.5 min)                   ║
║  🐢 Slowest Replier : Aman       (avg 56.9 min)                   ║
╠══════════════════════════════════════════════════════════════════╣
║  📊  AVG REPLY TIMES PER PERSON                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  Rahul       ███████████░░░░░░░  36.5 min            ║
║  Priya       █████████████░░░░░  43.0 min            ║
║  Neha        ████████████░░░░░░  40.9 min            ║
║  Aman        ██████████████████  56.9 min            ║
║  Karan       ███████████░░░░░░░  37.5 min            ║
║  Vikas       ██████████████░░░░  46.3 min            ║
╠══════════════════════════════════════════════════════════════════╣
║  🏜️   LONGEST SILENT STREAKS (conse

## 🎤 Feature 7 — Conversation Starters, Killers & Interaction Analysis

In [17]:
#  FEATURE 7 — CONVERSATION STARTERS & KILLERS
#  A "conversation starter" sends the first message of the day.
#  A "conversation killer" sends the last message of a conversation
#  after which no one responds for 4+ hours.
#  Interaction analysis shows who replies to whom.

def compute_conversation_roles(messages):
    """
    Computes:
        starters – {sender: count of times they sent first msg of a day}
        killers  – {sender: count of times their msg was followed by 4h silence}
    """
    starters = {name: 0 for name in PARTICIPANT_LIST}
    killers  = {name: 0 for name in PARTICIPANT_LIST}

    # Group by date to find daily first message
    seen_dates = {}
    for m in messages:
        if m["datetime_obj"] is None:
            continue
        date_key = m["datetime_obj"].date()
        if date_key not in seen_dates:
            seen_dates[date_key] = m["sender"]

    for sender in seen_dates.values():
        if sender in starters:
            starters[sender] += 1

    # Find conversation killers (silence >4h after their msg)
    SILENCE_THRESHOLD = 4 * 3600  # 4 hours in seconds
    for i in range(len(messages) - 1):
        curr = messages[i]
        nxt  = messages[i + 1]
        if curr["datetime_obj"] and nxt["datetime_obj"]:
            gap = (nxt["datetime_obj"] - curr["datetime_obj"]).total_seconds()
            if gap > SILENCE_THRESHOLD:
                sender = curr["sender"]
                if sender in killers:
                    killers[sender] += 1

    return starters, killers


def compute_interaction_matrix(messages, participant_list):
    """
    Builds a NumPy (n × n) matrix where cell [i, j] = number of times
    person i replied to person j (different sender, gap < 30 min).
    """
    n       = len(participant_list)
    pidx    = {name: i for i, name in enumerate(participant_list)}
    matrix  = np.zeros((n, n), dtype=int)

    for i in range(1, len(messages)):
        curr = messages[i]
        prev = messages[i - 1]
        if (curr["sender"] != prev["sender"] and
                curr["datetime_obj"] and prev["datetime_obj"]):
            gap = (curr["datetime_obj"] - prev["datetime_obj"]).total_seconds()
            if 0 < gap < 1800:  # within 30 minutes
                ri = pidx.get(curr["sender"])
                ci = pidx.get(prev["sender"])
                if ri is not None and ci is not None:
                    matrix[ri, ci] += 1

    return matrix


def print_interaction_analysis(starters, killers, int_matrix, participant_list):
    """Prints conversation starters, killers, and interaction matrix."""
    sorted_starters = sorted(starters.items(), key=lambda x: x[1], reverse=True)
    sorted_killers  = sorted(killers.items(),  key=lambda x: x[1], reverse=True)

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  🎤  CONVERSATION DYNAMICS".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    print("║  🚀  CONVERSATION STARTERS (first msg of the day)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    max_s = sorted_starters[0][1] if sorted_starters else 1
    for name, count in sorted_starters:
        bar = "█" * int(count / max_s * 18) + "░" * (18 - int(count / max_s * 18))
        print(f"║  {name:<10}  {bar}  {count:>3} times{'':>16}║")

    print("╠" + "═" * W + "╣")
    print("║  💀  CONVERSATION KILLERS (last msg before 4h silence)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")

    max_k = sorted_killers[0][1] if sorted_killers else 1
    for name, count in sorted_killers:
        bar = "█" * int(count / max_k * 18) + "░" * (18 - int(count / max_k * 18))
        print(f"║  {name:<10}  {bar}  {count:>3} times{'':>16}║")

    print("╠" + "═" * W + "╣")
    print("║  🔁  INTERACTION MATRIX (who replies to whom, ≤30 min)".ljust(W) + "║")
    print("╠" + "═" * W + "╣")
    header = "             " + "  ".join(f"{n[:4]:>4}" for n in participant_list)
    print(f"║  {header.ljust(W-2)}║")
    print("╠" + "═" * W + "╣")
    for i, name in enumerate(participant_list):
        row_str = "  ".join(f"{int(int_matrix[i, j]):>4}" for j in range(len(participant_list)))
        print(f"║  {name:<10}   {row_str.ljust(W-14)}║")
    print("╚" + "═" * W + "╝")


# ── RUN
STARTERS, KILLERS     = compute_conversation_roles(messages)
INT_MATRIX            = compute_interaction_matrix(messages, PARTICIPANT_LIST)
print_interaction_analysis(STARTERS, KILLERS, INT_MATRIX, PARTICIPANT_LIST)
print("\n✅  Interaction analysis complete.")



╔══════════════════════════════════════════════════════════════════╗
║  🎤  CONVERSATION DYNAMICS                                        ║
╠══════════════════════════════════════════════════════════════════╣
║  🚀  CONVERSATION STARTERS (first msg of the day)                ║
╠══════════════════════════════════════════════════════════════════╣
║  Aman        ██████████████████   58 times                ║
║  Rahul       ░░░░░░░░░░░░░░░░░░    2 times                ║
║  Priya       ░░░░░░░░░░░░░░░░░░    0 times                ║
║  Neha        ░░░░░░░░░░░░░░░░░░    0 times                ║
║  Karan       ░░░░░░░░░░░░░░░░░░    0 times                ║
║  Vikas       ░░░░░░░░░░░░░░░░░░    0 times                ║
╠══════════════════════════════════════════════════════════════════╣
║  💀  CONVERSATION KILLERS (last msg before 4h silence)           ║
╠══════════════════════════════════════════════════════════════════╣
║  Aman        ██████████████████   12 times                ║
║  Rahul       

## 🏆 Feature 8 — Activity Score & Leaderboard

In [18]:
#  FEATURE 8 — ACTIVITY SCORE & LEADERBOARD
#  Composite score blending: message count, unique active days,
#  word count, response speed, media shared.

def compute_activity_scores(messages, participant_stats, streaks,
                             avg_times, participant_list, total_days):
    """
    Computes a composite activity score (0-100) per person.
    Weights:
        40% – normalised message count
        20% – normalised active-day ratio
        15% – normalised word output
        15% – response speed (inverted, faster = higher)
        10% – media shared
    """
    scores = {}
    names  = participant_list

    # Raw values
    msg_counts  = np.array([participant_stats[n]["msg_count"]   for n in names], dtype=float)
    word_counts = np.array([participant_stats[n]["word_count"]  for n in names], dtype=float)
    media_cnts  = np.array([participant_stats[n]["media_count"] for n in names], dtype=float)

    # Active days = total_days - silent_total
    active_days = np.array([total_days - streaks[n]["silent_total"] for n in names], dtype=float)

    # Response speed: lower time = higher score; cap at 12h
    MAX_TIME = 43200.0
    resp_raw = np.array([min(avg_times.get(n, MAX_TIME), MAX_TIME) for n in names])
    resp_norm = 1 - (resp_raw / MAX_TIME)  # inverted & normalised

    def safe_norm(arr):
        mx = np.max(arr)
        return arr / mx if mx > 0 else arr

    msg_norm   = safe_norm(msg_counts)
    word_norm  = safe_norm(word_counts)
    media_norm = safe_norm(media_cnts)
    day_norm   = safe_norm(active_days)

    composite = (0.40 * msg_norm +
                 0.20 * day_norm +
                 0.15 * word_norm +
                 0.15 * resp_norm +
                 0.10 * media_norm) * 100

    for i, name in enumerate(names):
        scores[name] = round(float(composite[i]), 1)

    return scores


def print_leaderboard(scores, participant_list):
    """Prints an activity leaderboard."""
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    medals = ["🥇", "🥈", "🥉"] + ["  "] * 10

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  🏆  ACTIVITY SCORE LEADERBOARD".center(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  {'RANK':<5} {'NAME':<12} {'SCORE BAR':<22} {'SCORE':>7}  ║")
    print("╠" + "═" * W + "╣")

    max_score = sorted_scores[0][1] if sorted_scores else 100
    for i, (name, score) in enumerate(sorted_scores):
        bar_len = int(score / max_score * 20)
        bar     = "█" * bar_len + "░" * (20 - bar_len)
        medal   = medals[i]
        print(f"║  {medal} {i+1:<3} {name:<12} {bar}  {score:>6}/100  ║")

    print("╚" + "═" * W + "╝")

    return sorted_scores


# ── RUN
ACTIVITY_SCORES = compute_activity_scores(
    messages, participant_stats, STREAKS,
    AVG_TIMES, PARTICIPANT_LIST, total_days
)
print_leaderboard(ACTIVITY_SCORES, PARTICIPANT_LIST)
print("\n✅  Activity scores computed.")



╔══════════════════════════════════════════════════════════════════╗
║                   🏆  ACTIVITY SCORE LEADERBOARD                  ║
╠══════════════════════════════════════════════════════════════════╣
║  RANK  NAME         SCORE BAR                SCORE  ║
╠══════════════════════════════════════════════════════════════════╣
║  🥇 1   Rahul        ████████████████████    84.8/100  ║
║  🥈 2   Neha         █████████████████░░░    73.3/100  ║
║  🥉 3   Karan        █████████████████░░░    72.8/100  ║
║     4   Priya        ████████████████░░░░    72.0/100  ║
║     5   Aman         ██████████████░░░░░░    61.2/100  ║
║     6   Vikas        █████░░░░░░░░░░░░░░░    22.9/100  ║
╚══════════════════════════════════════════════════════════════════╝

✅  Activity scores computed.


## 🧬 Feature 9 — Personality Archetype Detection

In [19]:
#  FEATURE 9 — PERSONALITY ARCHETYPE DETECTION
#  Implements all 8 official archetypes + 1 bonus archetype.
#  Each person gets exactly ONE archetype (highest score wins).
#  Tie-breaking rule: if two people tie on the same archetype,
#  the one with the higher raw score keeps it; the other falls
#  back to their second-best archetype.

# Tie-breaking rule:
# If a participant qualifies for multiple archetypes,
# assign the archetype with the highest normalized score.
# If scores are equal, use the priority order defined below.

# ── SCORE FUNCTIONS

def score_spammer(name, messages):
    """
    THE SPAMMER – avg consecutive burst length.
    A "burst" is a sequence of messages from the same person
    with no one else in between.
    """
    bursts       = []
    current_len  = 0
    current_name = None

    for m in messages:
        if m["sender"] == name:
            if current_name == name:
                current_len += 1
            else:
                if current_name == name and current_len > 0:
                    bursts.append(current_len)
                current_name = name
                current_len  = 1
        else:
            if current_name == name and current_len > 0:
                bursts.append(current_len)
            current_name = m["sender"]
            current_len  = 1

    if current_name == name and current_len > 0:
        bursts.append(current_len)

    return sum(bursts) / len(bursts) if bursts else 0


def score_group_mom(name, messages):
    """
    THE GROUP MOM – total count of caring keyword occurrences.
    Checks multi-word phrases too.
    """
    count = 0
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        text_lower = m["text"].lower()
        for keyword in CARING_KEYWORDS:
            count += text_lower.count(keyword)
    return count


def score_night_owl(name, messages, heatmap_matrix, participant_list):
    """
    THE NIGHT OWL – % of messages sent between 23:00 and 04:59.
    Uses the pre-built NumPy heatmap for efficient calculation.
    """
    pidx = participant_list.index(name) if name in participant_list else None
    if pidx is None:
        return 0
    row          = heatmap_matrix[pidx]
    night_hours  = list(range(23, 24)) + list(range(0, 5))
    night_msgs   = int(np.sum(row[night_hours]))
    total_msgs   = int(np.sum(row))
    return (night_msgs / total_msgs * 100) if total_msgs > 0 else 0


def score_storyteller(name, messages):
    """
    THE STORYTELLER – average words per message (real messages only).
    """
    word_counts = []
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        word_counts.append(len(m["text"].split()))
    return sum(word_counts) / len(word_counts) if word_counts else 0


def score_drama_queen(name, messages):
    """
    THE DRAMA QUEEN – % of messages that are all-caps OR have 2+ exclamation marks.
    Short messages (<3 chars) are excluded from the all-caps check.
    """
    qualifying = 0
    total      = 0
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        total += 1
        text   = m["text"]
        alpha_only = "".join(c for c in text if c.isalpha())
        is_allcaps = (len(alpha_only) >= 3 and alpha_only.isupper())
        has_excl   = text.count("!") >= 2
        if is_allcaps or has_excl:
            qualifying += 1
    return (qualifying / total * 100) if total > 0 else 0


def score_ghost(name, streaks, total_days):
    """
    THE GHOST – % of days with zero messages.
    """
    return streaks[name]["silent_pct"]


def score_comedian(name, messages):
    """
    THE COMEDIAN – % of messages containing a comedy keyword.
    """
    comedy_msgs = 0
    total       = 0
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        total += 1
        text_lower = m["text"].lower()
        if any(kw in text_lower for kw in COMEDY_KEYWORDS):
            comedy_msgs += 1
    return (comedy_msgs / total * 100) if total > 0 else 0


def score_question_master(name, messages):
    """
    THE QUESTION MASTER – % of messages ending with '?'.
    """
    question_msgs = 0
    total         = 0
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        total += 1
        if m["text"].strip().endswith("?"):
            question_msgs += 1
    return (question_msgs / total * 100) if total > 0 else 0


def score_night_philosopher(name, messages):
    """
    BONUS ARCHETYPE — THE LATE-NIGHT PHILOSOPHER 🌙
    Sends messages containing introspective words ('life', 'time', 'meaning',
    'truth', 'feel', 'think', 'wonder', 'universe', 'dream', 'reality')
    after midnight (00:00 – 04:59).
    Score = count of such messages.
    """
    PHILO_WORDS = {"life", "time", "meaning", "truth", "feel", "think",
                   "wonder", "universe", "dream", "reality", "soul", "world",
                   "exist", "purpose", "why", "deep", "believe", "question"}
    count = 0
    for m in messages:
        if m["sender"] != name or m["is_media"] or m["is_deleted"]:
            continue
        if m["datetime_obj"] is None:
            continue
        hour = m["datetime_obj"].hour
        if 0 <= hour < 5:
            text_lower = m["text"].lower()
            words_set  = set(tokenize(text_lower))
            if words_set & PHILO_WORDS:
                count += 1
    return count


# ── ARCHETYPE ASSIGNMENT

ARCHETYPES = [
    ("THE SPAMMER",               "🔥"),
    ("THE GROUP MOM",             "💛"),
    ("THE NIGHT OWL",             "🦉"),
    ("THE STORYTELLER",           "📖"),
    ("THE DRAMA QUEEN",           "🎭"),
    ("THE GHOST",                 "👻"),
    ("THE COMEDIAN",              "😂"),
    ("THE QUESTION MASTER",       "❓"),
    ("THE LATE-NIGHT PHILOSOPHER","🌙"),  # BONUS
]

ARCHETYPE_NAMES = [a[0] for a in ARCHETYPES]


def compute_all_scores(participant_list, messages, heatmap_matrix,
                        streaks, total_days):
    """
    Returns a dict: {sender: {archetype_name: score}}
    """
    all_scores = {}

    for name in participant_list:
        all_scores[name] = {
            "THE SPAMMER"               : score_spammer(name, messages),
            "THE GROUP MOM"             : score_group_mom(name, messages),
            "THE NIGHT OWL"             : score_night_owl(name, messages, heatmap_matrix, participant_list),
            "THE STORYTELLER"           : score_storyteller(name, messages),
            "THE DRAMA QUEEN"           : score_drama_queen(name, messages),
            "THE GHOST"                 : score_ghost(name, streaks, total_days),
            "THE COMEDIAN"              : score_comedian(name, messages),
            "THE QUESTION MASTER"       : score_question_master(name, messages),
            "THE LATE-NIGHT PHILOSOPHER": score_night_philosopher(name, messages),
        }

    return all_scores


def assign_archetypes(all_scores, participant_list):
    """
    Exclusive assignment: each archetype can only be assigned to ONE person
    (the one with the highest score for it).
    Tie-breaking: the person with the higher raw score wins.
    Others fall back to their next-best unassigned archetype.
    """
    # Sort people by their top-score archetype
    # Strategy: iterate archetypes, assign to best scorer, then remove
    assigned       = {}   # name  → archetype
    assigned_to    = {}   # archetype → name (reverse)
    scores_copy    = {name: dict(s) for name, s in all_scores.items()}

    # Rank each person's archetype preferences
    person_prefs = {}
    for name in participant_list:
        prefs = sorted(scores_copy[name].items(), key=lambda x: x[1], reverse=True)
        person_prefs[name] = prefs   # ordered list of (archetype, score)

    unassigned = list(participant_list)

    # Iterate until everyone is assigned
    max_rounds = len(ARCHETYPE_NAMES) * len(participant_list)
    round_num  = 0
    while unassigned and round_num < max_rounds:
        round_num += 1
        # Each unassigned person "claims" their top available archetype
        claims = {}  # archetype → list of (name, score)
        for name in unassigned:
            for archetype, score in person_prefs[name]:
                if archetype not in assigned_to:
                    claims.setdefault(archetype, []).append((name, score))
                    break

        # Resolve: for each claimed archetype, winner = highest score
        for archetype, claimants in claims.items():
            winner = max(claimants, key=lambda x: x[1])
            assigned[winner[0]]  = archetype
            assigned_to[archetype] = winner[0]
            unassigned.remove(winner[0])

    # Fallback: if somehow unassigned (shouldn't happen), give "THE COMEDIAN"
    for name in unassigned:
        assigned[name] = "THE COMEDIAN"

    return assigned


def build_archetype_reason(name, archetype, all_scores, messages,
                            streaks, total_days, heatmap_matrix,
                            participant_list):
    """Returns a human-readable reason string for the archetype assignment."""
    s = all_scores[name]

    if archetype == "THE SPAMMER":
        return f"avg {s['THE SPAMMER']:.1f} msgs in a row"
    elif archetype == "THE GROUP MOM":
        return f"caring keyword score: {int(s['THE GROUP MOM'])}"
    elif archetype == "THE NIGHT OWL":
        return f"{s['THE NIGHT OWL']:.1f}% msgs between 23h–04h"
    elif archetype == "THE STORYTELLER":
        return f"avg {s['THE STORYTELLER']:.1f} words per message"
    elif archetype == "THE DRAMA QUEEN":
        return f"{s['THE DRAMA QUEEN']:.1f}% ALL-CAPS or multi-exclamation messages"
    elif archetype == "THE GHOST":
        silent = streaks[name]["silent_total"]
        return f"silent on {silent} of {total_days} days ({s['THE GHOST']:.0f}%)"
    elif archetype == "THE COMEDIAN":
        return f"{s['THE COMEDIAN']:.1f}% messages contain comedy words"
    elif archetype == "THE QUESTION MASTER":
        return f"{s['THE QUESTION MASTER']:.1f}% messages end with '?'"
    elif archetype == "THE LATE-NIGHT PHILOSOPHER":
        return f"{int(s['THE LATE-NIGHT PHILOSOPHER'])} philosophical late-night messages"
    return ""


def print_archetypes(assigned, all_scores, participant_list, messages,
                     streaks, total_days, heatmap_matrix):
    """Prints personality archetype cards for each participant."""
    emoji_map = {a[0]: a[1] for a in ARCHETYPES}

    W = 66
    print()
    print("╔" + "═" * W + "╗")
    print("║" + "  🧬  PERSONALITY ARCHETYPES  — GroupDNA".center(W) + "║")
    print("╠" + "═" * W + "╣")

    for name in participant_list:
        archetype  = assigned[name]
        icon       = emoji_map.get(archetype, "🏷️")
        reason     = build_archetype_reason(name, archetype, all_scores, messages,
                                             streaks, total_days, heatmap_matrix,
                                             participant_list)
        stats      = all_scores[name]
        n_msgs     = participant_stats[name]["msg_count"]
        pct        = n_msgs / len(messages) * 100
        # runner-up
        sorted_s   = sorted(stats.items(), key=lambda x: x[1], reverse=True)
        runner_up  = next((a for a, _ in sorted_s if a != archetype), "—")
        ru_icon    = emoji_map.get(runner_up, "")

        print("╠" + "═" * W + "╣")
        print(f"║  👤 {name.upper():<{W-5}}║")
        print(f"║  {icon}  {archetype:<{W-5}}║")
        print(f"║  📌 Reason   : {reason:<{W-17}}║")
        print(f"║  📩 Messages : {n_msgs:>5} ({pct:.1f}% of group){'':<{W-35}}║")
        night_p = stats["THE NIGHT OWL"]
        print(f"║  🌙 Night %  : {night_p:>5.1f}%  (msgs after 23:00){'':<{W-39}}║")
        avg_resp = AVG_TIMES.get(name, 0)
        fmt = (f"{avg_resp/60:.1f} min" if avg_resp < 3600
               else f"{avg_resp/3600:.1f} hrs") if avg_resp else "N/A"
        print(f"║  ⚡ Avg Reply: {fmt:<{W-16}}║")
        print(f"║  🥈 Runner-up: {ru_icon} {runner_up:<{W-18}}║")

    print("╠" + "═" * W + "╣")
    print("║  🌙 BONUS ARCHETYPE: THE LATE-NIGHT PHILOSOPHER".ljust(W) + "║")
    print("║     Sends introspective msgs (life, meaning, dream) after midnight.".ljust(W) + "║")
    print("╚" + "═" * W + "╝")


# ── RUN
ALL_SCORES = compute_all_scores(PARTICIPANT_LIST, messages,
                                HEATMAP_MATRIX, STREAKS, total_days)
ASSIGNED   = assign_archetypes(ALL_SCORES, PARTICIPANT_LIST)
print_archetypes(ASSIGNED, ALL_SCORES, PARTICIPANT_LIST,
                 messages, STREAKS, total_days, HEATMAP_MATRIX)
print("\n✅  Archetype detection complete.")



╔══════════════════════════════════════════════════════════════════╗
║               🧬  PERSONALITY ARCHETYPES  — GroupDNA              ║
╠══════════════════════════════════════════════════════════════════╣
╠══════════════════════════════════════════════════════════════════╣
║  👤 RAHUL                                                        ║
║  🦉  THE NIGHT OWL                                                ║
║  📌 Reason   : 13.4% msgs between 23h–04h                       ║
║  📩 Messages :   953 (30.0% of group)                               ║
║  🌙 Night %  :  13.4%  (msgs after 23:00)                           ║
║  ⚡ Avg Reply: 36.5 min                                          ║
║  🥈 Runner-up: 🔥 THE SPAMMER                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  👤 PRIYA                                                        ║
║  💛  THE GROUP MOM                                                ║
║  📌 Reason   : caring keyword scor

## 📋 Feature 10 — Final Screenshot-Ready Report

In [20]:
#  FEATURE 10 — FINAL CONSOLIDATED REPORT
#  The all-in-one screenshot-worthy report.
#  Run this cell to see the complete GroupDNA output.

def print_final_report():
    """Prints the complete GroupDNA analytics report."""

    W        = 68
    total    = len(messages)
    max_msgs = max(participant_stats[n]["msg_count"] for n in PARTICIPANT_LIST)
    emoji_map = {a[0]: a[1] for a in ARCHETYPES}

    def sep(char="═"):
        print("╠" + char * W + "╣")

    # ── HEADER
    print()
    print("╔" + "═" * W + "╗")
    print("║" + ("  🧬  GROUPDNA REPORT  ─  " + f'"{GROUP_NAME}"').center(W) + "║")
    print(f"║  {total_days} days  •  {total:,} messages  •  {len(PARTICIPANT_LIST)} members".center(W) + "║")
    print("╠" + "═" * W + "╣")
    print(f"║  Period      : {first_dt.strftime('%d %B %Y')} → {last_dt.strftime('%d %B %Y'):<35}║")

    busiest_day_str  = format_date_readable(max(date_counts, key=date_counts.get))
    busiest_day_cnt  = date_counts[max(date_counts, key=date_counts.get)]
    busiest_h        = max(hour_counts, key=hour_counts.get)
    print(f"║  Busiest Day : {busiest_day_str} ({busiest_day_cnt} messages){'':>16}║")
    print(f"║  Busiest Hour: {busiest_h:02d}:00 – {busiest_h+1:02d}:00{'':>42}║")

    # ── MESSAGES PER PERSON
    sep()
    print("║" + "  📩  MESSAGES PER PERSON".ljust(W) + "║")
    sep()
    for name in PARTICIPANT_LIST:
        cnt  = participant_stats[name]["msg_count"]
        pct  = cnt / total * 100
        bl   = int(cnt / max_msgs * 20)
        bar  = "█" * bl + ("░" * (20 - bl) if bl < 20 else "")
        if cnt < 50:
            bar = "."
        print(f"║  {name:<10}  {bar:<20}  {cnt:>4}  ({pct:>4.1f}%){'':>9}║")

    # ── HEATMAP
    sep()
    print("║" + "  🌡️   ACTIVITY HEATMAP  (hour of day, cols 00→23)".ljust(W) + "║")
    sep()
    print(f"║  {'NAME':<12} 00 02 04 06 08 10 12 14 16 18 20 22{'':>4}║")
    sep("─")

    SHADES = ['.', '░', '▒', '▓', '█']
    for i, name in enumerate(PARTICIPANT_LIST):
        row     = HEATMAP_MATRIX[i]
        row_max = int(np.max(row))
        shaded  = []
        for h in range(0, 24, 2):
            val   = int(row[h]) + int(row[h+1]) if h+1 < 24 else int(row[h])
            rm2   = row_max * 2 if h+1 < 24 else row_max
            if rm2 == 0 or val == 0:
                shaded.append('.')
            else:
                ratio = val / rm2
                if ratio <= 0.25:  shaded.append('░')
                elif ratio <= 0.5: shaded.append('▒')
                elif ratio <= 0.75:shaded.append('▓')
                else:              shaded.append('█')
        shade_str = "  ".join(shaded)
        archetype = ASSIGNED.get(name, "")
        tag = " ← " + archetype.replace("THE ","") if name == PARTICIPANT_LIST[0] else ""
        print(f"║  {name:<12} {shade_str}  {tag.ljust(14)}║")

    # ── TOP WORDS
    sep()
    print("║" + "  💬  THIS GROUP'S FAVOURITE WORDS".ljust(W) + "║")
    sep()
    top15    = top_n(GLOBAL_FREQ, 10)
    max_wc   = top15[0][1] if top15 else 1
    for word, count in top15:
        bl = int(count / max_wc * 20)
        bar = "█" * bl + "░" * (20 - bl)
        print(f"║  {word:<12}  {bar}  {count:>4}{'':>9}║")

    # ── RESPONSE PATTERNS
    sep()
    print("║" + "  ⏱️   RESPONSE PATTERNS".ljust(W) + "║")
    sep()

    valid_times = {n: t for n, t in AVG_TIMES.items() if t != float('inf')}
    if valid_times:
        fastest_n = min(valid_times, key=valid_times.get)
        slowest_n = max(valid_times, key=valid_times.get)
        def _fmt(t):
            if t < 60:   return f"{t:.0f} sec"
            elif t < 3600: return f"{t/60:.1f} min"
            else:         return f"{t/3600:.1f} hrs"
        print(f"║  ⚡ Fastest replier : {fastest_n} ({_fmt(valid_times[fastest_n])}){'':>29}║")
        print(f"║  🐢 Slowest replier : {slowest_n} ({_fmt(valid_times[slowest_n])}){'':>29}║")

    # ── SILENT STREAKS
    sep()
    print("║" + "  🏜️   LONGEST SILENT STREAKS".ljust(W) + "║")
    sep()
    sorted_streak = sorted(PARTICIPANT_LIST,
                           key=lambda n: STREAKS[n]["max_streak"],
                           reverse=True)
    for name in sorted_streak:
        s   = STREAKS[name]
        ms  = s["max_streak"]
        rng = ""
        if s["streak_start"] and ms > 0:
            rng = f"({s['streak_start'].strftime('%d %b')} – {s['streak_end'].strftime('%d %b')})"
        note = "(never went silent ✨)" if ms == 0 else rng
        print(f"║  {name:<10}  {ms:>2} days  {note:<42}║")

    # ── ARCHETYPES
    sep()
    print("║" + "  🧬  PERSONALITY ARCHETYPES".ljust(W) + "║")
    sep()
    for name in PARTICIPANT_LIST:
        archetype = ASSIGNED[name]
        icon      = emoji_map.get(archetype, "🏷️")
        reason    = build_archetype_reason(name, archetype, ALL_SCORES, messages,
                                           STREAKS, total_days, HEATMAP_MATRIX,
                                           PARTICIPANT_LIST)
        print(f"║  {name:<10}  →  {icon} {archetype:<25}  {reason[:20]:<20}║")

    # ── FOOTER
    sep()
    print("║" + "  🏆  ACTIVITY LEADERBOARD".ljust(W) + "║")
    sep()
    sorted_s = sorted(ACTIVITY_SCORES.items(), key=lambda x: x[1], reverse=True)
    medals   = ["🥇","🥈","🥉"] + ["  "] * 10
    for i, (name, score) in enumerate(sorted_s):
        bl  = int(score / 100 * 18)
        bar = "█" * bl + "░" * (18 - bl)
        print(f"║  {medals[i]} {name:<10}  {bar}  {score:>5}/100{'':>11}║")

    sep()
    print("║" + "  Generated by GroupDNA  •  Built with Python + NumPy".center(W) + "║")
    print("║" + "  No pandas. No matplotlib. No regex. Just fundamentals.".center(W) + "║")
    print("╚" + "═" * W + "╝")


print_final_report()



╔════════════════════════════════════════════════════════════════════╗
║              🧬  GROUPDNA REPORT  ─  "Hostel Bois 4ever"            ║
            ║  60 days  •  3,174 messages  •  6 members             ║
╠════════════════════════════════════════════════════════════════════╣
║  Period      : 01 April 2024 → 30 May 2024                        ║
║  Busiest Day : 04 May 2024 (76 messages)                ║
║  Busiest Hour: 18:00 – 19:00                                          ║
╠════════════════════════════════════════════════════════════════════╣
║  📩  MESSAGES PER PERSON                                            ║
╠════════════════════════════════════════════════════════════════════╣
║  Rahul       ████████████████████   953  (30.0%)         ║
║  Priya       ███████████████░░░░░   718  (22.6%)         ║
║  Neha        █████████████░░░░░░░   635  (20.0%)         ║
║  Aman        ██████████░░░░░░░░░░   490  (15.4%)         ║
║  Karan       ███████░░░░░░░░░░░░░   354  (11.2%)     

## 🃏 Feature 11 — Individual Summary Cards

In [21]:
#  FEATURE 11 — INDIVIDUAL SUMMARY CARDS
#  One card per participant: key stats + archetype + top words

def print_summary_cards(participant_list):
    emoji_map = {a[0]: a[1] for a in ARCHETYPES}
    total     = len(messages)

    for name in participant_list:
        s         = participant_stats[name]
        archetype = ASSIGNED[name]
        icon      = emoji_map.get(archetype, "🏷️")
        reason    = build_archetype_reason(name, archetype, ALL_SCORES, messages,
                                           STREAKS, total_days, HEATMAP_MATRIX,
                                           participant_list)
        night_p   = ALL_SCORES[name]["THE NIGHT OWL"]
        real_msgs = s["msg_count"] - s["media_count"] - s["deleted_count"]
        avg_words = s["word_count"] / real_msgs if real_msgs > 0 else 0
        pct       = s["msg_count"] / total * 100
        top5w     = top_n(PERSON_FREQ.get(name, {}), 5)
        words_str = ", ".join(f"{w}({c})" for w, c in top5w)
        silent    = STREAKS[name]["silent_total"]
        avg_t     = AVG_TIMES.get(name, float('inf'))
        def _fmt(t):
            if t == float('inf'): return "N/A"
            if t < 60:   return f"{t:.0f}s"
            elif t < 3600: return f"{t/60:.1f}m"
            else:         return f"{t/3600:.1f}h"

        W = 50
        print()
        print("  ╔" + "═" * W + "╗")
        print(f"  ║  👤  {name.upper():<{W-5}}║")
        print("  ╠" + "═" * W + "╣")
        print(f"  ║  {icon}  {archetype:<{W-5}}║")
        print(f"  ║  📌  {reason:<{W-5}}║")
        print("  ╠" + "═" * W + "╣")
        print(f"  ║  📩 Total Messages  : {s['msg_count']:>5} ({pct:.1f}%){'':>{W-31}}║")
        print(f"  ║  📷 Media Shared    : {s['media_count']:>5}{'':>{W-24}}║")
        print(f"  ║  🗑️ Deleted Msgs    : {s['deleted_count']:>5}{'':>{W-24}}║")
        print(f"  ║  📝 Avg Words/Msg   : {avg_words:>5.1f}{'':>{W-24}}║")
        print(f"  ║  🌙 Night Activity  : {night_p:>5.1f}%{'':>{W-25}}║")
        print(f"  ║  ⚡ Avg Reply Time  : {_fmt(avg_t):<{W-22}}║")
        print(f"  ║  🏜️ Silent Days     : {silent:>5} of {total_days}{'':>{W-27}}║")
        print("  ╠" + "═" * W + "╣")
        print(f"  ║  🔤 Top Words: {words_str[:35]:<{W-15}}║")
        print("  ╚" + "═" * W + "╝")


print_summary_cards(PARTICIPANT_LIST)
print("\n✅  Summary cards printed.")



  ╔══════════════════════════════════════════════════╗
  ║  👤  RAHUL                                        ║
  ╠══════════════════════════════════════════════════╣
  ║  🦉  THE NIGHT OWL                                ║
  ║  📌  13.4% msgs between 23h–04h                   ║
  ╠══════════════════════════════════════════════════╣
  ║  📩 Total Messages  :   953 (30.0%)                   ║
  ║  📷 Media Shared    :     7                          ║
  ║  🗑️ Deleted Msgs    :     6                          ║
  ║  📝 Avg Words/Msg   :   2.6                          ║
  ║  🌙 Night Activity  :  13.4%                         ║
  ║  ⚡ Avg Reply Time  : 36.5m                       ║
  ║  🏜️ Silent Days     :     0 of 60                       ║
  ╠══════════════════════════════════════════════════╣
  ║  🔤 Top Words: bhai(159), scene(144), kya(133), ya║
  ╚══════════════════════════════════════════════════╝

  ╔══════════════════════════════════════════════════╗
  ║  👤  PRIYA                          

## 🧪 Edge Case Validation & Self-Test

In [24]:
#  EDGE CASE VALIDATION
#  Tests the parser on synthetic edge-case inputs to confirm
#  robustness on unusual inputs.

def run_edge_case_tests():
    """Runs built-in edge case tests and prints pass/fail."""
    print("🧪  Running edge case tests ...\n")
    tests_passed = 0
    tests_total  = 0

    def assert_test(condition, description):
        nonlocal tests_passed, tests_total
        tests_total += 1
        status = "✅  PASS" if condition else "❌  FAIL"
        print(f"    {status}  {description}")
        if condition:
            tests_passed += 1

    # 1. System message → skip
    system_line = "01/04/24, 01:13 - Priya created group \"Hostel Bois 4ever\""
    assert_test(parse_line(system_line) is None,
                "System message returns None")

    # 2. Real message parsed correctly
    real_line = "12/04/24, 23:14 - Rahul: bhai aaj kya scene"
    parsed    = parse_line(real_line)
    assert_test(parsed is not None and parsed["sender"] == "Rahul",
                "Real message parsed with correct sender")
    assert_test(parsed is not None and parsed["text"] == "bhai aaj kya scene",
                "Real message text extracted correctly")

    # 3. Media omitted
    media_line = "12/04/24, 14:22 - Rahul: <Media omitted>"
    m_parsed   = parse_line(media_line)
    assert_test(m_parsed is not None and m_parsed["text"] == "<Media omitted>",
                "Media omitted message parsed")

    # 4. Deleted message
    del_line = "13/04/24, 09:15 - Neha: This message was deleted"
    d_parsed = parse_line(del_line)
    assert_test(d_parsed is not None and d_parsed["text"] == "This message was deleted",
                "Deleted message parsed")

    # 5. Date detection
    assert_test(is_date_line("12/04/24, 23:14 - Rahul: hi"), "Date line detection works")
    assert_test(not is_date_line("This is a continuation line"), "Non-date line detected")
    assert_test(not is_date_line(""), "Empty line not flagged as date line")

    # 6. Stop word removal
    tokens = tokenize("I am going to the market")
    assert_test("market" in tokens and "the" not in tokens and "to" not in tokens,
                "Stop words removed correctly")

    # 7. Punctuation stripping
    assert_test(strip_punctuation("bhai!") == "bhai",
                "Trailing punctuation stripped")
    assert_test(strip_punctuation("...yaar?") == "yaar",
                "Leading punctuation stripped")

    # 8. NumPy matrix integrity
    total_from_np = int(np.sum(HEATMAP_MATRIX))
    assert_test(total_from_np == len(messages),
                f"NumPy matrix total ({total_from_np}) matches message count ({len(messages)})")

    # 9. All participants have archetypes
    all_assigned = all(name in ASSIGNED for name in PARTICIPANT_LIST)
    assert_test(all_assigned, "All participants have an assigned archetype")

    # 10. Archetypes are exclusive
    archetype_values = list(ASSIGNED.values())
    unique_check     = len(archetype_values) == len(set(archetype_values))
    assert_test(unique_check, "Each archetype assigned to at most one person (exclusive)")

    print()
    print(f"  Result: {tests_passed}/{tests_total} tests passed")
    if tests_passed == tests_total:
        print("  🎉  All edge case tests passed!")
    else:
        print("  ⚠️   Some tests failed. Review the logic above.")


run_edge_case_tests()


🧪  Running edge case tests ...

    ✅  PASS  System message returns None
    ✅  PASS  Real message parsed with correct sender
    ✅  PASS  Real message text extracted correctly
    ✅  PASS  Media omitted message parsed
    ✅  PASS  Deleted message parsed
    ✅  PASS  Date line detection works
    ✅  PASS  Non-date line detected
    ✅  PASS  Empty line not flagged as date line
    ✅  PASS  Stop words removed correctly
    ✅  PASS  Trailing punctuation stripped
    ✅  PASS  Leading punctuation stripped
    ✅  PASS  NumPy matrix total (3174) matches message count (3174)
    ✅  PASS  All participants have an assigned archetype
    ✅  PASS  Each archetype assigned to at most one person (exclusive)

  Result: 14/14 tests passed
  🎉  All edge case tests passed!


## 📝 Constraint Summary


**Constraint Summary (Project Rules Followed):**
- ✅ No `pandas` — used plain dicts and lists  
- ✅ No `matplotlib` — all charts are Unicode text-based  
- ✅ No `collections.Counter` — built custom word-frequency dict  
- ✅ No `re` (regex) — all parsing via `str.split()`, `str.startswith()`, etc.  
- ✅ NumPy used for heatmap matrix, nightly stats, activity scores  
- ✅ Works on any WhatsApp `.txt` export — fully dynamic, no hardcoded names

**Built by:** Alzafar Khan | **Batch:** June 1 |
